<a href="https://colab.research.google.com/github/KinzaAsif2456/discoverey/blob/main/work/notebooks/w04_baseline_score.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-07 — Baseline Action Score and Top-20 Review

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [16]:
import os, sys, subprocess
import pandas as pd
import numpy as np

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/flyrank-bih/flyrank-ml-internship-starter"
REPO_DIR = "flyrank-ml-internship-starter"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
else:
    while not os.path.isdir("data/raw") and os.getcwd() != "/":
        os.chdir("..")

print("Working dir:", os.getcwd())
assert os.path.exists("data/raw/content_refresh_anonymized.csv"), "starter CSV not found — are you at the repo root?"
print("Starter data found. You're ready.")

Working dir: /content/discoverey/flyrank-ml-internship-starter/flyrank-ml-internship-starter/flyrank-ml-internship-starter
Starter data found. You're ready.


## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

In [17]:
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
df["is_declining_label"] = (df["trend_direction"] == "down").astype(int)
base_rate = df["is_declining_label"].mean()
print(f"rows: {len(df)}, base decline rate: {base_rate:.3f}")

rows: 30000, base decline rate: 0.542


In [18]:
# --- Signal 1: staleness (behind FlyRank's refresh flags) ---
order = ["0-30", "31-90", "91-180", "181+"]
staleness_table = (df.groupby("freshness_tier")
                      .agg(n=("is_declining_label", "size"),
                           decline_rate=("is_declining_label", "mean"))
                      .reindex(order))
print(staleness_table)

                    n  decline_rate
freshness_tier                     
0-30            20480      0.511377
31-90             175      0.588571
91-180           9171      0.611057
181+              174      0.471264


In [19]:
# --- Signal 2: CTR-vs-position (behind the CTR-fix logic) ---
df["low_ctr_flag"] = (
    (df["impressions_90d"] >= 500) &
    (df["avg_position"] > 0) & (df["avg_position"] <= 20) &
    (df["ctr"] < 0.5)
)

ctr_table = (df.groupby("low_ctr_flag")
               .agg(n=("is_declining_label", "size"),
                    decline_rate=("is_declining_label", "mean")))
print(ctr_table)

                  n  decline_rate
low_ctr_flag                     
False         20241      0.501062
True           9759      0.627113


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [20]:
df["stale_flag"] = df["freshness_tier"].isin(["91-180", "181+"])

df["score"] = df["stale_flag"].astype(int) + df["low_ctr_flag"].astype(int)

def reason_code(row):
    if row["low_ctr_flag"]:
        return "CTR_UNDERPERFORM_FOR_POSITION"
    if row["stale_flag"]:
        return "STALE_CONTENT"
    return "NO_STRONG_SIGNAL"

df["reason_code"] = df.apply(reason_code, axis=1)

def action_label(s):
    if s == 2: return "REFRESH_NOW"
    if s == 1: return "REVIEW"
    return "NO_ACTION"

df["action"] = df["score"].apply(action_label)

def confidence_note(s):
    if s == 2: return "High — meets both signal conditions"
    if s == 1: return "Medium — meets one signal condition"
    return "Low — no strong signal present"

df["confidence"] = df["score"].apply(confidence_note)

# tiebreak equal scores by visibility (impressions), so ranking is meaningful
queue = df.sort_values(["score", "impressions_90d"], ascending=[False, False]).reset_index(drop=True)
queue["rank"] = queue.index + 1

precision_at_50 = queue.head(50)["is_declining_label"].mean()
print(f"Precision@50: {precision_at_50:.3f}  |  base rate: {base_rate:.3f}")

import os
os.makedirs("work/outputs", exist_ok=True)
out_cols = ["content_id", "client_id", "rank", "score", "reason_code", "action", "confidence", "is_declining_label"]
queue[out_cols].to_csv("work/outputs/baseline_action_score.csv", index=False)
print("wrote", len(queue), "rows")

Precision@50: 0.360  |  base rate: 0.542
wrote 30000 rows


In [21]:
!head work/outputs/baseline_action_score.csv

content_id,client_id,rank,score,reason_code,action,confidence,is_declining_label
content_5fe46e04994d,client_4e07408562,1,2,CTR_UNDERPERFORM_FOR_POSITION,REFRESH_NOW,High — meets both signal conditions,1
content_cb112fce36be,client_19581e27de,2,2,CTR_UNDERPERFORM_FOR_POSITION,REFRESH_NOW,High — meets both signal conditions,1
content_36ff89c8214e,client_19581e27de,3,2,CTR_UNDERPERFORM_FOR_POSITION,REFRESH_NOW,High — meets both signal conditions,0
content_c21024970297,client_19581e27de,4,2,CTR_UNDERPERFORM_FOR_POSITION,REFRESH_NOW,High — meets both signal conditions,0
content_c8e9d6ab9013,client_19581e27de,5,2,CTR_UNDERPERFORM_FOR_POSITION,REFRESH_NOW,High — meets both signal conditions,1
content_d17681677e69,client_19581e27de,6,2,CTR_UNDERPERFORM_FOR_POSITION,REFRESH_NOW,High — meets both signal conditions,0
content_a7427266c305,client_19581e27de,7,2,CTR_UNDERPERFORM_FOR_POSITION,REFRESH_NOW,High — meets both signal conditions,0
content_c5063073d048,client_6208ef0f77,8,2,CTR_UNDERPERFOR

## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [22]:
review_cols = ["content_id", "rank", "score", "reason_code", "action", "confidence",
               "days_since_last_update", "avg_position", "ctr", "impressions_90d", "is_declining_label"]
print(queue[review_cols].head(20).to_string(index=False))

          content_id  rank  score                   reason_code      action                          confidence  days_since_last_update  avg_position  ctr  impressions_90d  is_declining_label
content_5fe46e04994d     1      2 CTR_UNDERPERFORM_FOR_POSITION REFRESH_NOW High — meets both signal conditions                     104           4.2 0.14           517715                   1
content_cb112fce36be     2      2 CTR_UNDERPERFORM_FOR_POSITION REFRESH_NOW High — meets both signal conditions                     104           5.6 0.16           309910                   1
content_36ff89c8214e     3      2 CTR_UNDERPERFORM_FOR_POSITION REFRESH_NOW High — meets both signal conditions                     104           7.3 0.05           295097                   0
content_c21024970297     4      2 CTR_UNDERPERFORM_FOR_POSITION REFRESH_NOW High — meets both signal conditions                     104           5.1 0.41           211366                   0
content_c8e9d6ab9013     5      2 CTR_UN

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

In [23]:
print("Columns used in scoring: freshness_tier, avg_position, impressions_90d, ctr")
print("trend_direction / trend_pct used only to build is_declining_label (the label) —",
      "never inside stale_flag, low_ctr_flag, score, reason_code, or action.")

borderline = queue[queue["score"] == 1].sort_values("impressions_90d").head(3)
print(borderline[review_cols])

Columns used in scoring: freshness_tier, avg_position, impressions_90d, ctr
trend_direction / trend_pct used only to build is_declining_label (the label) — never inside stale_flag, low_ctr_flag, score, reason_code, or action.
                 content_id   rank  score    reason_code  action  \
15470  content_36e7b91747fa  15471      1  STALE_CONTENT  REVIEW   
15471  content_3b50b0c27334  15472      1  STALE_CONTENT  REVIEW   
15482  content_75bbe9161b5d  15483      1  STALE_CONTENT  REVIEW   

                                confidence  days_since_last_update  \
15470  Medium — meets one signal condition                     211   
15471  Medium — meets one signal condition                     104   
15482  Medium — meets one signal condition                     102   

       avg_position  ctr  impressions_90d  is_declining_label  
15470           0.0  0.0                1                   0  
15471          23.0  0.0                1                   0  
15482           9.0  0.0    

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.